In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math
from mpl_toolkits.mplot3d import Axes3D
from scipy.integrate import solve_ivp
import ipywidgets
import plotly.graph_objects as go
import rasterio
from matplotlib.colors import LightSource
from ipywidgets import interact, FloatSlider, IntSlider
import plotly.io as pio
from scipy.spatial import Delaunay
import pyvista as pv
import rasterio
from scipy.interpolate import RegularGridInterpolator
from plotly.subplots import make_subplots
from rasterio.enums import Resampling
from rasterio.windows import Window
pio.renderers.default = 'vscode'
from utils import *
import trimesh
import pyvista as pv
pv.set_jupyter_backend('trame')

In [ ]:
# --- CONFIGURAZIONE MESH FORNITA DA TE ---
vertices = np.array([
    [1.0, 0.0, 0.0],  
    [0.0, 1.0, 0.0],  
    [0.0, 0.0, 1.0]   
], dtype=float)

triangoli = np.array([[0, 1, 2]], dtype=int)

mesh_trimesh = trimesh.Trimesh(vertices=vertices, faces=triangoli)
mesh_trimesh.fix_normals()
normale_analitica = mesh_trimesh.face_normals[0]

faces_pv = np.array([3, 0, 1, 2], dtype=int)
mesh_pv = pv.PolyData(mesh_trimesh.vertices, faces_pv)

p_start_tx = np.array([0.5, 0.5, 1.5])
# direzione_tx = np.array([0.0, 0.0, -1.0]) 
direzione_tx = -np.array([0.5774, 0.5774, 0.5774])

# --- ESECUZIONE PIPELINE DI RAY TRACING ---
p_end = p_start_tx + (direzione_tx * 10.0)
p_intersezioni, id_facce = mesh_pv.ray_trace(p_start_tx, p_end)

print(p_intersezioni)
if len(p_intersezioni) > 0:
    # Estraiamo il primo punto di impatto trovato
    punto_contatto = p_intersezioni[0]
    
    # Calcolo riflessione specchiata vettoriale
    normale_superficie = normale_analitica.copy()
    if np.dot(direzione_tx, normale_superficie) > 0:
        normale_superficie = -normale_superficie
        
    direzione_riflessa = direzione_tx - 2 * np.dot(direzione_tx, normale_superficie) * normale_superficie
    direzione_riflessa /= np.linalg.norm(direzione_riflessa)
    
    # --- CORRETTO: Calcolo analitico dinamico spostato qui (all'interno del blocco di successo) ---
    z_atteso_dinamico = 1.0 - p_start_tx[0] - p_start_tx[1]
    atteso_punto_dinamico = np.array([p_start_tx[0], p_start_tx[1], z_atteso_dinamico])
    
    print("--- CONFRONTO RISULTATI (Carta e Penna vs Codice) ---")
    print(f"Normale Mesh:      {np.round(normale_superficie, 4)}")
    print(f"Punto Impatto:     {np.round(punto_contatto, 4)} (Atteso: {np.round(atteso_punto_dinamico, 4)})")
    print(f"Vettore Riflesso:  {np.round(direzione_riflessa, 4)}")
    
    # Test di validità con tolleranza numerica infinitesima
    print("\n✅ OK: Il Raytracer calcola l'angolo di rimbalzo spaziale alla perfezione!")
else:
    print("❌ Errore: Il raggio ha mancato il triangolo.")


In [ ]:
# ==============================================================================
# 1. CONFIGURAZIONE DELLA MESH (4 Vertici, 2 Triangoli)
# ==============================================================================
vertices = np.array([
    [1.0, 0.0, 0.0],  # V0
    [0.0, 1.0, 0.0],  # V1
    [0.0, 0.0, 1.0],  # V2
    [1.0, 1.0, 1.0]   # V3
], dtype=float)

triangoli = np.array([
    [0, 1, 2],        # Faccia 0
    [0, 1, 3]         # Faccia 1 (Configurazione aggiornata da te)
], dtype=int)

# Configurazione Trimesh (senza fix_normals per evitare l'errore networkx)
mesh_trimesh = trimesh.Trimesh(vertices=vertices, faces=triangoli)
mesh_trimesh.fix_normals()
normale_analitica = mesh_trimesh.face_normals[0] 

# Configurazione PyVista per visualizzare entrambi i triangoli
faces_pv = np.array([
    3, 0, 1, 2,  # Primo triangolo (3 vertici: 0, 1, 2)
    3, 0, 1, 3   # Secondo triangolo (3 vertici: 0, 1, 3)
], dtype=int)
mesh_pv = pv.PolyData(mesh_trimesh.vertices, faces_pv)

# ==============================================================================
# 2. DEFINIZIONE DEL RAGGIO INCIDENTE INIZIALE
# ==============================================================================
p_start_tx = np.array([0.3, 0.3, 0.5])
direzione_tx = np.array([0.0, 0.0, -1.0]) 

# Normalizzazione matematica del vettore direzione incidente
direzione_tx /= np.linalg.norm(direzione_tx)

print("--- VERIFICA MATEMATICA INIZIALE ---")
print(f"Posizione TX iniziale: {p_start_tx}")
print(f"Direzione Raggio Iniziale: {direzione_tx}")
print(f"Normale Faccia 0 (Trimesh): {np.round(normale_analitica, 4)}")

# ==============================================================================
# 3. PIPELINE DI RAY TRACING RICORSIVA (MULTI-BOUNCE)
# ==============================================================================
max_bounce = 2
punti_traiettoria = [p_start_tx.copy()]  # Lista per salvare la cronologia dei punti per il plot
raggio_disperso = True                   # Flag per capire se l'ultimo raggio esce dalla scena

print("\n--- AVVIO SIMULAZIONE RIMBALZI ---")

for bounce in range(max_bounce):
    # Calcoliamo il punto di arrivo virtuale del raggio per PyVista
    p_end = p_start_tx + (direzione_tx * 10.0)
    p_intersezioni, id_facce = mesh_pv.ray_trace(p_start_tx, p_end)
    
    if len(p_intersezioni) > 0:
        punto_contatto = p_intersezioni[0]
        
        # Identificazione sicura dell'indice della faccia colpita
        if hasattr(id_facce, "__len__") and len(id_facce) > 0:
            idx_faccia = int(id_facce[0])
        else:
            idx_faccia = int(id_facce)
        
        # Estrazione e normalizzazione della normale della faccia colpita
        normale_superficie = mesh_trimesh.face_normals[idx_faccia].copy()
        normale_superficie /= np.linalg.norm(normale_superficie)
        
        # Orientamento della normale in senso opposto al raggio incidente
        if np.dot(direzione_tx, normale_superficie) > 0:
            normale_superficie = -normale_superficie
            
        # Calcolo della direzione di riflessione speculare ideale
        direzione_riflessa = direzione_tx - 2 * np.dot(direzione_tx, normale_superficie) * normale_superficie
        direzione_riflessa /= np.linalg.norm(direzione_riflessa)
        
        print(f"\n[Rimbalzo {bounce}] -> Impatto Rilevato!")
        print(f"  Faccia Colpita (ID): {idx_faccia}")
        print(f"  Punto d'Impatto: {np.round(punto_contatto, 4)}")
        print(f"  Vettore Riflesso: {np.round(direzione_riflessa, 4)}")
        
        # Registriamo il punto d'impatto nella storia del raggio
        punti_traiettoria.append(punto_contatto.copy())
        
        # --- FIX SELF-INTERSECTION BIAS ---
        # Spostiamo l'origine del prossimo raggio leggermente staccata dalla superficie
        EPSILON_BIAS = 1e-4
        p_start_tx = punto_contatto + (direzione_riflessa * EPSILON_BIAS)
        direzione_tx = direzione_riflessa
    else:
        print(f"\n[Rimbalzo {bounce}] -> Il raggio non ha colpito nessuna superficie. Esce dai confini.")
        # Se non colpisce nulla al primo colpo, salviamo la fine del raggio nel vuoto
        punti_traiettoria.append(p_start_tx + (direzione_tx * 1.5))
        raggio_disperso = False
        break

# Se il raggio ha completato tutti i rimbalzi colpendo sempre qualcosa, 
# prounghiamo graficamente l'ultimo raggio riflesso verso l'infinito
if raggio_disperso:
    punti_traiettoria.append(p_start_tx + (direzione_tx * 1.5))

# ==============================================================================
# 4. VISUALIZZAZIONE 3D INTERATTIVA (PyVista Plotter)
# ==============================================================================
plotter = pv.Plotter()

# 1. Disegna la mesh poligonale
plotter.add_mesh(mesh_pv, color="cyan", opacity=0.5, show_edges=True, edge_color="black", label="Mesh 3D")

# 2. Disegna il punto di trasmissione originario (Sfera blu)
plotter.add_mesh(pv.Sphere(radius=0.02, center=punti_traiettoria[0]), color="blue", label="Sorgente TX")

# 3. Ciclo grafico per tracciare tutti i segmenti calcolati
for i in range(len(punti_traiettoria) - 1):
    p_inizio = punti_traiettoria[i]
    p_fine = punti_traiettoria[i+1]
    
    if i == 0:
        # Segmento 0: Raggio primario emesso dalla sorgente (Rosso)
        plotter.add_mesh(pv.Line(p_inizio, p_fine), color="red", line_width=5, label="Raggio Incidente")
    else:
        # Segmenti successivi: Raggi che hanno rimbalzato (Giallo)
        lbl = "Raggi Rimbalzati" if i == 1 else None
        plotter.add_mesh(pv.Line(p_inizio, p_fine), color="yellow", line_width=5, label=lbl)
    
    # Se il punto corrente è un impatto reale sulla mesh, disegna un marcatore sferico (Nero)
    if i < len(punti_traiettoria) - 2:
        plotter.add_mesh(pv.Sphere(radius=0.015, center=p_fine), color="black")

# Impostazioni estetiche della camera e della finestra
plotter.add_legend(bcolor=None, face="line")
plotter.add_axes()
plotter.camera_position = [(2.0, 2.0, 2.0), (0.5, 0.5, 0.5), (0.0, 0.0, 1.0)]
plotter.show()


In [ ]:
# ==============================================================================
# 1. FUNZIONE MATEMATICA: INTERSEZIONE RAGGIO-SFERA (RICEVITORE RX)
# ==============================================================================
def interseca_sfera_rx(origine, direzione, centro_rx, raggio_rx, max_dist):
    """
    Ritorna la distanza 't' del primo impatto con la sfera se il raggio la attraversa,
    altrimenti ritorna None. Considera valido l'impatto solo entro 'max_dist'.
    """
    v = centro_rx - origine
    proiezione = np.dot(v, direzione)
    
    # Calcolo del discriminante (geometria sotto la radice)
    dist_quadrata_centro = np.dot(v, v) - proiezione**2
    raggio_quadrato = raggio_rx**2
    
    if dist_quadrata_centro > raggio_quadrato:
        return None  # Il raggio passa fuori dalla sfera
        
    # Calcolo della distanza di ingresso nella sfera
    dist_interna = np.sqrt(raggio_quadrato - dist_quadrata_centro)
    t = proiezione - dist_interna
    
    # Il punto deve essere davanti al raggio e prima del prossimo ostacolo/limite
    if 0.0 < t < max_dist:
        return t
    return None

# ==============================================================================
# 2. CONFIGURAZIONE DELLA MESH (4 Vertici, 2 Triangoli)
# ==============================================================================
vertices = np.array([
    [1.0, 0.0, 0.0],  # V0
    [0.0, 1.0, 0.0],  # V1
    [0.0, 0.0, 1.0],  # V2
    [1.0, 1.0, 1.0]   # V3
], dtype=float)

triangoli = np.array([
    [0, 1, 2],        # Faccia 0
    [0, 1, 3]         # Faccia 1
], dtype=int)

mesh_trimesh = trimesh.Trimesh(vertices=vertices, faces=triangoli)
faces_pv = np.array([3, 0, 1, 2, 3, 0, 1, 3], dtype=int)
mesh_pv = pv.PolyData(mesh_trimesh.vertices, faces_pv)

# ==============================================================================
# 3. DEFINIZIONE DEL TRASMETTITORE (TX) E DEL RICEVITORE (RX)
# ==============================================================================
p_start_tx = np.array([0.3, 0.3, 0.5])
direzione_tx = np.array([0.0, 0.0, -1.0]) 
direzione_tx /= np.linalg.norm(direzione_tx)

# Configurazione Ricevitore Sferico (RX)
# Posizionato lungo la traiettoria del primo raggio riflesso per intercettarlo
centro_rx = np.array([0.5, 0.5, 0.3])
raggio_rx = 0.04

# ==============================================================================
# 4. PIPELINE DI RAY TRACING CON INTERCETTAZIONE RX
# ==============================================================================
max_bounce = 3
punti_traiettoria = [p_start_tx.copy()]
raggio_ricevuto = False
raggio_disperso = True

print("--- AVVIO SIMULAZIONE CON RICEVITORE ---")

for bounce in range(max_bounce):
    dist_massima_ricerca = 10.0
    p_end = p_start_tx + (direzione_tx * dist_massima_ricerca)
    
    # 1. Tracciamento contro la mesh geometrica
    p_intersezioni, id_facce = mesh_pv.ray_trace(p_start_tx, p_end)
    
    # Calcoliamo a quale distanza si trova l'ostacolo della mesh (se presente)
    dist_ostacolo = dist_massima_ricerca
    punto_mesh = None
    if len(p_intersezioni) > 0:
        punto_mesh = p_intersezioni[0]
        dist_ostacolo = np.linalg.norm(punto_mesh - p_start_tx)
        
    # 2. Test di intercettazione con la Sfera RX prima che il raggio colpisca il prossimo ostacolo
    t_rx = interseca_sfera_rx(p_start_tx, direzione_tx, centro_rx, raggio_rx, dist_ostacolo)
    
    if t_rx is not None:
        # Il raggio ha intersecato la sfera del ricevitore!
        punto_ricezione = p_start_tx + (direzione_tx * t_rx)
        punti_traiettoria.append(punto_ricezione.copy())
        print(f"\n🎯 [Rimbalzo {bounce}] INGRESSO SFERA RX RILEVATO!")
        print(f"  Coordinated di Ricezione: {np.round(punto_ricezione, 4)}")
        raggio_ricevuto = True
        raggio_disperso = False
        break # Ferma la simulazione perché il segnale è stato catturato
        
    # 3. Se non ha intersecato l'RX, procediamo con il normale impatto sulla mesh
    if punto_mesh is not None:
        if hasattr(id_facce, "__len__") and len(id_facce) > 0:
            idx_faccia = int(id_facce[0])
        else:
            idx_faccia = int(id_facce)
            
        normale_superficie = mesh_trimesh.face_normals[idx_faccia].copy()
        normale_superficie /= np.linalg.norm(normale_superficie)
        
        if np.dot(direzione_tx, normale_superficie) > 0:
            normale_superficie = -normale_superficie
            
        direzione_riflessa = direzione_tx - 2 * np.dot(direzione_tx, normale_superficie) * normale_superficie
        direzione_riflessa /= np.linalg.norm(direzione_riflessa)
        
        print(f"\n[Rimbalzo {bounce}] -> Passa accanto all'RX e colpisce la Mesh (Faccia {idx_faccia})")
        punti_traiettoria.append(punto_mesh.copy())
        
        # Bias anti-autocollisione per il prossimo ciclo
        EPSILON_BIAS = 1e-4
        p_start_tx = punto_mesh + (direzione_riflessa * EPSILON_BIAS)
        direzione_tx = direzione_riflessa
    else:
        print(f"\n[Rimbalzo {bounce}] -> Il raggio si perde nello spazio.")
        punti_traiettoria.append(p_start_tx + (direzione_tx * 1.5))
        raggio_disperso = False
        break

if raggio_disperso and not raggio_ricevuto:
    punti_traiettoria.append(p_start_tx + (direzione_tx * 1.5))

# ==============================================================================
# 5. VISUALIZZAZIONE 3D INTERATTIVA (PyVista Plotter)
# ==============================================================================
plotter = pv.Plotter()

# Disegna Mesh, Sorgente TX (Blu) e il Ricevitore RX (Sfera Magenta semitrasparente)
plotter.add_mesh(mesh_pv, color="cyan", opacity=0.4, show_edges=True, edge_color="black", label="Mesh")
plotter.add_mesh(pv.Sphere(radius=0.02, center=punti_traiettoria[0]), color="blue", label="Sorgente TX")
plotter.add_mesh(pv.Sphere(radius=raggio_rx, center=centro_rx), color="magenta", opacity=0.5, label="Ricevitore RX")

# Ciclo grafico per disegnare i segmenti
for i in range(len(punti_traiettoria) - 1):
    p_inizio = punti_traiettoria[i]
    p_fine = punti_traiettoria[i+1]
    
    if i == 0:
        plotter.add_mesh(pv.Line(p_inizio, p_fine), color="red", line_width=5, label="Raggio Incidente")
    else:
        lbl = "Raggio Riflesso" if i == 1 else None
        # Se l'ultimo punto è l'RX, coloriamo l'ultimo tratto di verde (segnale ricevuto)
        colore_tratto = "green" if (raggio_ricevuto and i == len(punti_traiettoria)-2) else "yellow"
        if colore_tratto == "green": lbl = "Segnale Ricevuto"
        plotter.add_mesh(pv.Line(p_inizio, p_fine), color=colore_tratto, line_width=5, label=lbl)
        
    # Disegna i punti di impatto intermedi sulla mesh (sfere nere)
    if i < len(punti_traiettoria) - 2 or (len(punti_traiettoria) > 2 and not raggio_ricevuto and i == len(punti_traiettoria) - 2):
        plotter.add_mesh(pv.Sphere(radius=0.012, center=p_fine), color="black")

plotter.add_legend(bcolor=None, face="line")
plotter.add_axes()
plotter.show()